# Módulo 6 — Práctica: montaje del esqueleto y refactorización a `pipeline/`

Pasos 1 y 2 de la práctica del **Día 10** (ver [`README.md`](../README.md)):
crear la estructura de carpetas y mover las funciones de los Módulos 3–5 a
`pipeline/`, parametrizadas desde `config.py`.

In [1]:
import os
import shutil

BASE = "proyecto_demo"
for carpeta in [
    f"{BASE}/datos/entrada", f"{BASE}/datos/wrfout", f"{BASE}/datos/observaciones",
    f"{BASE}/resultados/mapas", f"{BASE}/resultados/series", f"{BASE}/resultados/validacion",
    f"{BASE}/pipeline", f"{BASE}/notebooks",
]:
    os.makedirs(carpeta, exist_ok=True)

# Los generadores sinteticos se copian dentro de pipeline/ para que los
# modulos del pipeline puedan importarlos sin depender de rutas externas.
shutil.copy("synth_data_era5.py", f"{BASE}/pipeline/synth_data_era5.py")
shutil.copy("synth_data_wrfout.py", f"{BASE}/pipeline/synth_data_wrfout.py")

print("Estructura creada:")
for root, dirs, files in os.walk(BASE):
    nivel = root.replace(BASE, "").count(os.sep)
    indent = "  " * nivel
    print(f"{indent}{os.path.basename(root)}/")

Estructura creada:
proyecto_demo/
  pipeline/
  notebooks/
  datos/
    entrada/
    wrfout/
    observaciones/
  resultados/
    mapas/
    series/
    validacion/


## `config.py` — parametrización centralizada del caso

In [2]:
config_contenido = '''# config.py -- parametros del caso de estudio (Modulo 6)

CASO = {
    "fecha_inicio": "2026-06-01",
    "fecha_fin": "2026-06-02",
    "area": [5, -80, -15, -70],  # [norte, oeste, sur, este]
    "estacion_validacion": {"nombre": "Demo", "lat": -9.5, "lon": -76.0},
    "seed": 100,
}

RUTAS = {
    "entrada": "proyecto_demo/datos/entrada",
    "wrfout": "proyecto_demo/datos/wrfout",
    "observaciones": "proyecto_demo/datos/observaciones",
    "resultados_mapas": "proyecto_demo/resultados/mapas",
    "resultados_series": "proyecto_demo/resultados/series",
    "resultados_validacion": "proyecto_demo/resultados/validacion",
}
'''

with open(f"{BASE}/config.py", "w") as f:
    f.write(config_contenido)
print(f"Guardado: {BASE}/config.py")

Guardado: proyecto_demo/config.py


## `pipeline/descarga.py` — funciones del Módulo 3, parametrizadas

In [3]:
descarga_contenido = '''# pipeline/descarga.py (Modulo 3, refactorizado)
import os
import pandas as pd
import xarray as xr
from synth_data_era5 import era5_like_dataset


def descargar_era5(fecha_inicio, fecha_fin, area, destino, seed=0):
    norte, oeste, sur, este = area
    periodos = int((pd.Timestamp(fecha_fin) - pd.Timestamp(fecha_inicio)).total_seconds() // 3600) + 1
    ds = era5_like_dataset(
        lat_first=norte, lat_last=sur, lon_first=oeste, lon_last=este,
        nlat=15, nlon=15, start=fecha_inicio, periods=max(periodos, 1), seed=seed,
    )
    ds.to_netcdf(destino)
    return destino


def verificar_integridad(archivo, cobertura_esperada=None, tamano_minimo_kb=1):
    if not os.path.exists(archivo):
        return False, f"El archivo {archivo} no existe."
    tamano_kb = os.path.getsize(archivo) / 1024
    if tamano_kb < tamano_minimo_kb:
        return False, f"Archivo sospechosamente pequeno ({tamano_kb:.1f} KB)."
    try:
        ds = xr.open_dataset(archivo)
    except Exception as e:
        return False, f"No se pudo abrir el archivo: {type(e).__name__}: {e}"
    if cobertura_esperada and "time_min_horas" in cobertura_esperada:
        n_tiempos = ds.sizes.get("time", 1)
        if n_tiempos < cobertura_esperada["time_min_horas"]:
            return False, f"Cobertura insuficiente: {n_tiempos} pasos."
    return True, "OK"


def obtener_entrada_wrf(config, rutas):
    os.makedirs(rutas["entrada"], exist_ok=True)
    fecha_inicio = config["fecha_inicio"]
    fecha_fin = config["fecha_fin"]
    nombre = f"{fecha_inicio}_{fecha_fin}.nc"
    destino = os.path.join(rutas["entrada"], nombre)
    descargar_era5(config["fecha_inicio"], config["fecha_fin"], config["area"], destino, seed=config.get("seed", 0))
    horas = int((pd.Timestamp(config["fecha_fin"]) - pd.Timestamp(config["fecha_inicio"])).total_seconds() // 3600)
    ok, msg = verificar_integridad(destino, {"time_min_horas": max(horas, 1)})
    return (True, destino) if ok else (False, msg)


def series_gfs_en_puntos(ciclo, horas, puntos, carpeta):
    # Herbie (Modulo 3): baja solo T2m de cada hora de pronostico y extrae el
    # punto de grilla mas cercano a cada estacion. puntos = {nombre: (lat, lon)}.
    # Devuelve una tabla en degC: una fila por hora, una columna por estacion.
    from herbie import Herbie
    filas = {}
    for fxx in horas:
        h = Herbie(ciclo, model="gfs", product="pgrb2.0p25", fxx=fxx,
                   save_dir=carpeta, verbose=False)
        ds = h.xarray(":TMP:2 m above ground")
        tiempo = pd.Timestamp(ds["valid_time"].values)
        filas[tiempo] = {
            nombre: float(ds["t2m"].sel(latitude=lat, longitude=lon % 360, method="nearest")) - 273.15
            for nombre, (lat, lon) in puntos.items()
        }
    return pd.DataFrame.from_dict(filas, orient="index")
'''

with open(f"{BASE}/pipeline/descarga.py", "w") as f:
    f.write(descarga_contenido)
print(f"Guardado: {BASE}/pipeline/descarga.py")

Guardado: proyecto_demo/pipeline/descarga.py


## `pipeline/visualizacion.py` — funciones del Módulo 4, parametrizadas

In [4]:
visualizacion_contenido = '''# pipeline/visualizacion.py (Modulo 4, refactorizado)
import xarray as xr
import xwrf
import numpy as np


def abrir_wrfout(archivo):
    raw = xr.open_dataset(archivo)
    return raw.xwrf.postprocess().xwrf.destagger()


def precipitacion_horaria(ds):
    return (ds["RAINC"] + ds["RAINNC"]).diff("Time")


def punto_mas_cercano(ds, lat, lon):
    dist2 = (ds["XLAT"] - lat) ** 2 + (ds["XLONG"] - lon) ** 2
    iy, ix = np.unravel_index(np.argmin(dist2.values), dist2.shape)
    return iy, ix


def serie_temperatura_en_punto(ds, lat, lon):
    iy, ix = punto_mas_cercano(ds, lat, lon)
    return ds["T2"].isel(y=iy, x=ix) - 273.15


def guardar_mapa_temperatura(ds, ruta_salida, paso_tiempo=0):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    t2c = ds["T2"].isel(Time=paso_tiempo) - 273.15
    fig = plt.figure(figsize=(6, 5))
    ax = plt.axes(projection=ccrs.PlateCarree())
    malla = ax.pcolormesh(ds["XLONG"], ds["XLAT"], t2c, transform=ccrs.PlateCarree(),
                           cmap="RdYlBu_r", shading="auto")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.8)
    plt.colorbar(malla, ax=ax, label="T2 (degC)")
    fig.tight_layout()
    fig.savefig(ruta_salida, dpi=100)
    plt.close(fig)
    return ruta_salida
'''

with open(f"{BASE}/pipeline/visualizacion.py", "w") as f:
    f.write(visualizacion_contenido)
print(f"Guardado: {BASE}/pipeline/visualizacion.py")

Guardado: proyecto_demo/pipeline/visualizacion.py


## `pipeline/validacion.py` — funciones del Módulo 5, parametrizadas

In [5]:
validacion_contenido = '''# pipeline/validacion.py (Modulo 5, refactorizado)
import numpy as np
import pandas as pd


def calcular_mae(s, o):
    return float(np.mean(np.abs(s - o)))


def calcular_rmse(s, o):
    return float(np.sqrt(np.mean((s - o) ** 2)))


def calcular_bias(s, o):
    return float(np.mean(s - o))


def calcular_kge(s, o):
    r = float(np.corrcoef(s, o)[0, 1])
    beta = float(np.mean(s) / np.mean(o))
    alpha = float(np.std(s) / np.std(o))
    kge = 1 - np.sqrt((r - 1) ** 2 + (beta - 1) ** 2 + (alpha - 1) ** 2)
    return kge, {"r": r, "beta": beta, "alpha": alpha}


def leer_observaciones(ruta, variable="temp_C"):
    obs = pd.read_csv(ruta, parse_dates=["tiempo"], index_col="tiempo")
    return obs[variable].dropna().rename("observado")


def leer_senamhi(ruta, variable="Temperatura (°C)"):
    # SENAMHI: fecha en 4 columnas, hora LOCAL (UTC-5) y "S/D" = sin dato
    d = pd.read_csv(ruta, sep=";", dtype=str)
    tiempo = pd.to_datetime(d["Año"] + "-" + d["Mes"] + "-" + d["Día"] + " " + d["Hora"])
    valores = pd.to_numeric(d[variable], errors="coerce")  # "S/D" -> NaN
    serie = pd.Series(valores.values, index=tiempo + pd.Timedelta(hours=5), name="observado")
    return serie.dropna()


def evaluar_variable(casos, nombre_variable="temperatura"):
    filas = []
    for nombre_estacion, (obs, sim) in casos.items():
        kge, comp = calcular_kge(sim.values, obs.values)
        filas.append({
            "estacion": nombre_estacion, "variable": nombre_variable,
            "MAE": calcular_mae(sim.values, obs.values),
            "RMSE": calcular_rmse(sim.values, obs.values),
            "bias": calcular_bias(sim.values, obs.values),
            "r": comp["r"], "KGE": kge,
        })
    return pd.DataFrame(filas).set_index("estacion")
'''

with open(f"{BASE}/pipeline/validacion.py", "w") as f:
    f.write(validacion_contenido)
print(f"Guardado: {BASE}/pipeline/validacion.py")

Guardado: proyecto_demo/pipeline/validacion.py


## Verificación rápida: los módulos son importables

In [6]:
import sys
sys.path.insert(0, BASE)
sys.path.insert(0, f"{BASE}/pipeline")

import importlib
import descarga, visualizacion, validacion
for m in [descarga, visualizacion, validacion]:
    importlib.reload(m)

print("Funciones en descarga.py:    ", [f for f in dir(descarga) if not f.startswith('_')])
print("Funciones en visualizacion.py:", [f for f in dir(visualizacion) if not f.startswith('_')])
print("Funciones en validacion.py:  ", [f for f in dir(validacion) if not f.startswith('_')])

Funciones en descarga.py:     ['descargar_era5', 'era5_like_dataset', 'obtener_entrada_wrf', 'os', 'pd', 'series_gfs_en_puntos', 'verificar_integridad', 'xr']
Funciones en visualizacion.py: ['abrir_wrfout', 'guardar_mapa_temperatura', 'np', 'precipitacion_horaria', 'punto_mas_cercano', 'serie_temperatura_en_punto', 'xr', 'xwrf']
Funciones en validacion.py:   ['calcular_bias', 'calcular_kge', 'calcular_mae', 'calcular_rmse', 'evaluar_variable', 'leer_observaciones', 'leer_senamhi', 'np', 'pd']


## Cierre

Entregable parcial: la carpeta `proyecto_demo/` completa. Continuar con
[`03_practica_ejecucion_completa.ipynb`](03_practica_ejecucion_completa.ipynb).